In [ ]:
import cohere
import os
import numpy as np
import pandas as pd
from numpy.typing import NDArray

COHERE_API_KEY = os.getenv("COHERE_API_KEY")
co = cohere.ClientV2(COHERE_API_KEY)


# Import all data from the scifact
df_train = pd.read_csv("./data/scifact/qrels/train.tsv", sep='\t')
df_test = pd.read_csv("./data/scifact/qrels/test.tsv", sep='\t')
df_corpus = pd.read_json("./data/scifact/corpus.jsonl", lines=True)
df_queries = pd.read_json("./data/scifact/queries.jsonl", lines=True)

display(df_train.info())
# display(df_test.info())
display(df_corpus.info())
display(df_queries.info())

display(df_corpus.head())
display(df_queries.head())

# def calc_cosine_similarity(doc1, doc2):
#     return np.dot(doc1,doc2) / ( np.linalg.norm(doc1) * np.linalg.norm(doc2))

def calculate_vectorized_similarity(query_embed: NDArray, corpus_embed: NDArray):
    return (query_embed @ corpus_embed.T / (np.linalg.norm(query_embed) * np.linalg.norm(corpus_embed.T)))[0]

<class 'pandas.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   query-id   919 non-null    int64
 1   corpus-id  919 non-null    int64
 2   score      919 non-null    int64
dtypes: int64(3)
memory usage: 21.7 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 5183 entries, 0 to 5182
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   _id       5183 non-null   int64 
 1   title     5183 non-null   str   
 2   text      5183 non-null   str   
 3   metadata  5183 non-null   object
dtypes: int64(1), object(1), str(2)
memory usage: 7.6+ MB


None

<class 'pandas.DataFrame'>
RangeIndex: 1109 entries, 0 to 1108
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   _id       1109 non-null   int64 
 1   text      1109 non-null   str   
 2   metadata  1109 non-null   object
dtypes: int64(1), object(1), str(1)
memory usage: 122.6+ KB


None

,_id,title,text,metadata
0,4983,Microstructural development of human newborn c...,Alterations of the architecture of cerebral wh...,{}
1,5836,Induction of myelodysplasia by myeloid-derived...,Myelodysplastic syndromes (MDS) are age-depend...,{}
2,7912,"BC1 RNA, the transcript from a master gene for...",ID elements are short interspersed elements (S...,{}
3,18670,The DNA Methylome of Human Peripheral Blood Mo...,DNA methylation plays an important role in bio...,{}
4,19238,The human myelin basic protein gene is include...,Two human Golli (for gene expressed in the oli...,{}


,_id,text,metadata
0,0,0-dimensional biomaterials lack inductive prop...,{}
1,2,1 in 5 million in UK have abnormal PrP positiv...,"{'13734012': [{'sentences': [4], 'label': 'CON..."
2,4,1-1% of colorectal cancer patients are diagnos...,{}
3,6,10% of sudden infant death syndrome (SIDS) dea...,{}
4,9,32% of liver transplantation programs required...,"{'44265107': [{'sentences': [15], 'label': 'SU..."


# Baseline comparison before Cohere Reranker


In [31]:
from cohere import EmbedInputType 

class MockResponse():
    embeddings = []
    def __init__(self, array):
        self.embeddings = array

def embed_cohere_batched(model, input_type: EmbedInputType, text_or_corpus: list, max_texts_per_call = 96):
    embeds = []
    TOTAL_CHUNKS = round(len(text_or_corpus) / max_texts_per_call)
    for n_call in range(1,TOTAL_CHUNKS):
        chunk = text_or_corpus[(n_call-1) * max_texts_per_call :n_call * max_texts_per_call]
        # print(n_call, len(chunk))
        # cohere_embed_response = co.embed(model=model,
        #                         input_type=input_type,
        #                         texts=text_or_corpus,
        #                         embedding_types=['float'])
        cohere_embed_response = MockResponse([np.random.normal(0,1,1024) for x in range(max_texts_per_call)])
        embeds += cohere_embed_response.embeddings
        # print(np.random.normal(0,1,1024))
        # break;
    return np.array(embeds)



# do embedding
embed_models = [
    # embed-v5.0-pro
    # Our most capable embedding model, for text and images. Higher quality than embed-v5.0-fast, at higher latency.
    # Text, Images, Mixed texts/images (i.e. PDFs)
    # One of ‘[256, 512, 768, 1024, 1536, 2048 (default)]‘
    # 128k
    # Cosine Similarity, Dot Product Similarity, Euclidean Distance
    # Embed
    'embed-v5.0-pro',
    # embed-v5.0-fast
    # A faster, lighter version of embed-v5.0-pro. Allows for text and images to be classified or turned into embeddings
    # Text, Images, Mixed texts/images (i.e. PDFs)
    # One of ‘[256, 512, 768, 1024, 1536, 2048 (default)]‘
    # 128k
    # Cosine Similarity, Dot Product Similarity, Euclidean Distance
    # Embed
    'embed-v5.0-fast'
]
model = embed_models[-1]

# saving locally since it's a small dataset
EMBED_CORPUS_FILE_NAME = model + '_cache_embedded_corpus.npy'
EMBED_QUERIES_FILE_NAME = model + '_cache_embedded_queries.npy'

embedded_corpus = np.array([])

try:
    embedded_corpus = np.fromfile(EMBED_CORPUS_FILE_NAME)
except:
    print(f"No embed file {EMBED_CORPUS_FILE_NAME}, fetching from cohere")

embedded_queries = np.array([])
try:
    embedded_queries = np.fromfile(EMBED_QUERIES_FILE_NAME)
except:
    print(f"No embed file {EMBED_QUERIES_FILE_NAME}, fetching from cohere")

    
# print(embed_cohere_batched(model,"search_document", list(df_corpus['text'])))
print(embedded_corpus.shape)
print(embedded_queries.shape)
# Maximum number of texts per call is 96.
if len(embedded_corpus) == 0:
    # embedded_corpus = co.embed(model=model,
    #                         input_type="search_document",
    #                         texts=list(df_corpus['text']),
    #                         embedding_types=['float'])
    embedded_corpus = embed_cohere_batched(model,"search_document", list(df_corpus['text']))
    print(embedded_corpus.shape)
    if len(embedded_corpus):
        np.save(EMBED_CORPUS_FILE_NAME, np.array(embedded_corpus))

if len(embedded_queries) == 0:
    embedded_queries = embed_cohere_batched(model,"search_query", list(df_queries['text']))
    print(embedded_queries.shape)                            
    # cache with numpy the results of all the embeds...
    if len(embedded_queries):
        np.save(EMBED_QUERIES_FILE_NAME, np.array(embedded_queries))


No embed file embed-v5.0-fast_cache_embedded_corpus.npy, fetching from cohere
No embed file embed-v5.0-fast_cache_embedded_queries.npy, fetching from cohere
(0,)
(0,)
(5088, 1024)
(1056, 1024)


# Cohere Reranker

In [ ]:
from cohere import V2RerankResponse
# rerank
rerank_models = [
    # 10k docs max
    'rerank-v5.0-pro',
    'rerank-v5.0-fast'
]
q = ""
docs = []
ks = [1, 3, 5, 10, 100]

def validate_result(response:V2RerankResponse):
    return response

for k in ks:
    response = co.rerank(model=rerank_models[-1],
                         query=q,
                         documents=docs,
                         top_n=k
                         )
    break


# Results Local Vs Cohere Reranker vs Others
